# (Không bắt buộc) Phục vụ adapter bằng vLLM trên Colab T4 và gọi qua OpenAI-compatible API

**Mục tiêu:** chạy `vllm.entrypoints.openai.api_server` với base `Qwen/Qwen2.5-0.5B-Instruct` + LoRA `ft=$OUT/full`
(`--dtype half`, `--tool-call-parser hermes`), rồi dùng **cùng** `training/predict_toolcall.py` nhưng với
`--backend openai` để dự đoán 50 bản ghi, chấm bằng `eval_toolcall.py`, và đo **p50 latency**. Đây đúng là đường đi
mà backend Chat System dùng để gọi SLM trong sản xuất.

**Cảnh báo trước khi bắt đầu**
- `pip install vllm` mất **vài phút** và thường **đổi phiên bản `torch`** → Colab yêu cầu *Restart session*.
  Sau khi restart: chạy lại Cell 1 và Cell 3, **bỏ qua** cell cài vLLM (đã cài rồi) và Cell 2 (không cần).
- Triệu chứng torch không khớp: `ImportError: … undefined symbol …`, `RuntimeError: … compiled with CUDA X but torch Y`,
  hoặc `vllm` import được nhưng `transformers`/`bitsandbytes` báo lỗi. Không pin thủ công gì trong notebook này
  (thử phiên bản mới nhất); nếu vẫn lỗi sau một lần restart → dùng mục **Dự phòng** ở cuối.
- Nếu bạn còn cần notebook 01/02 trong cùng phiên, hãy làm notebook này **sau cùng** hoặc trong phiên riêng.

### Cell 1 — Thiết lập (ô chung của mọi notebook, copy nguyên văn từ `notebooks/_setup_snippet.md`)
Trên Colab: đọc `GITHUB_TOKEN` từ **Secrets** (biểu tượng chìa khóa ở thanh bên trái, bật *Notebook access*),
`git clone` repo private `thanhhao98/ChatSystem` (bỏ qua nếu đã có) rồi `chdir` vào đó. Trên máy cá nhân: đi lên từ
thư mục hiện tại đến khi gặp `docs/contracts/cli.md`. Ô đặt các biến `REPO`, `GIT_SHA`, `IN_COLAB`, `AUTHOR` và hàm
`run(cmd)`. Kaggle: token đọc từ *Add-ons → Secrets*. **Không bao giờ** in token hay `!cat .git/config` vào output.

In [ ]:
# --- Thiết lập (Colab + local) --------------------------------------------------------------
# Colab : read GITHUB_TOKEN from Secrets, clone the private repo (skip if present), chdir into it.
# Local : walk up from the current directory until the repo root (docs/contracts/cli.md) is found.
# Sets REPO (Path), GIT_SHA, IN_COLAB, AUTHOR and a run() helper that calls repo scripts.
import os, shlex, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
MARKER = "docs/contracts/cli.md"          # exists at the root of every checkout

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _github_token():
    # Colab Secrets -> Kaggle Secrets -> environment variable. Never print the value.
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    return os.environ["GITHUB_TOKEN"]


if IN_COLAB or IN_KAGGLE:
    try:
        _token = _github_token()
    except Exception as e:  # SecretNotFoundError / NotebookAccessError / KeyError
        raise RuntimeError(
            "Thiếu secret GITHUB_TOKEN. Colab: biểu tượng chìa khoá (Secrets) -> Add new secret: "
            "Name = GITHUB_TOKEN, Value = Personal access token (classic, scope repo) của tài khoản collaborator "
            "trên thanhhao98/ChatSystem, bật 'Notebook access'. Kaggle: Add-ons -> Secrets -> GITHUB_TOKEN. "
            "Rồi chạy lại ô này.") from e
    if not Path("ChatSystem").exists():
        _r = subprocess.run(["git", "clone", "--quiet", f"https://{_token}@{REPO_HTTPS}", "ChatSystem"],
                            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone thất bại: " + _r.stderr.replace(_token, "<token>"))
    os.chdir("ChatSystem")
    del _token
else:
    _here = Path.cwd().resolve()
    for _cand in [_here, *_here.parents]:
        if (_cand / MARKER).exists():
            os.chdir(_cand)
            break
    else:
        raise FileNotFoundError(f"Không tìm thấy gốc repo (không có {MARKER}) khi đi lên từ {_here}. "
                                "Mở notebook từ bên trong thư mục ChatSystem đã clone.")

REPO = Path.cwd()


def _git(*args):
    # Small helper: run a git command in REPO and return stdout ("" on any failure).
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""


GIT_SHA = _git("rev-parse", "--short", "HEAD") or "no-git"
AUTHOR = os.environ.get("GITHUB_USER") or _git("config", "user.name") or "điền tên"


def run(cmd):
    # Run a repo script (list of args), echo the command, stream its output, return CompletedProcess.
    shown = " ".join(shlex.quote(c) for c in cmd).replace(shlex.quote(sys.executable), "python", 1)
    print("$ " + shown)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    print(f"[exit code = {p.returncode}]")
    return p


print(f"REPO      = {REPO}")
print(f"git HEAD  = {GIT_SHA}")
print(f"python    = {sys.version.split()[0]} · Colab = {IN_COLAB} · Kaggle = {IN_KAGGLE} · author = {AUTHOR}")

### Cell 2 — Cài thư viện theo pins và kiểm tra GPU
Dòng đầu tiên in ra trên Colab miễn phí phải là:

```
Tesla T4 · 15 GB · (7, 5) · USE_BF16=False → fp16
```

- `(7, 5)` là *compute capability*; T4 < 8 nên **fp16**. Script huấn luyện tự chọn dtype theo đúng quy tắc này
  (biến môi trường `FORCE_FP16=1` ép fp16 trên GPU mới hơn để tái lập điều kiện T4).
- Nếu thấy `NO CUDA GPU`: *Runtime → Change runtime type → T4 GPU* rồi chạy lại từ Cell 1.
- Các phiên bản in ra phải trùng `requirements-train.txt`; nếu Colab đổi phiên bản `torch` thì ghi vào comment
  của task và mở issue kèm dòng phiên bản (pins được kiểm tra lại qua PR), **không** tự sửa pins.

In [ ]:
# Cell 2 — install the pinned training stack (Colab/Kaggle only) and probe the GPU.
import os
IN_KAGGLE = bool(globals().get("IN_KAGGLE")) or ((not IN_COLAB) and os.path.isdir("/kaggle/working"))
if IN_COLAB or IN_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Local run: skipping pip install (use your own virtualenv built from requirements-train.txt).")

import importlib

import torch

FORCE_FP16 = os.environ.get("FORCE_FP16") == "1"
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
    GPU_CAP = torch.cuda.get_device_capability(0)
    # dtype is chosen by compute capability, NOT by torch.cuda.is_bf16_supported():
    # the T4 (7,5) only emulates bf16 and the training script would be slow / unstable.
    USE_BF16 = GPU_CAP[0] >= 8 and not FORCE_FP16
    GPU_LINE = (f"{GPU_NAME} · {GPU_GIB:.0f} GB · {GPU_CAP} · USE_BF16={USE_BF16} "
                f"→ {'bf16' if USE_BF16 else 'fp16'}")
else:
    GPU_NAME, GPU_GIB, GPU_CAP, USE_BF16 = None, 0.0, None, False
    GPU_LINE = "NO CUDA GPU (Colab: Runtime → Change runtime type → T4 GPU)"
print(GPU_LINE)

VERSIONS = {}
for _mod in ("torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate"):
    try:
        VERSIONS[_mod] = importlib.import_module(_mod).__version__
    except Exception as exc:  # ImportError, or a broken CUDA extension on CPU-only hosts
        VERSIONS[_mod] = f"not installed ({type(exc).__name__})"
    print(f"{_mod:<13} {VERSIONS[_mod]}")
VERSIONS_LINE = " · ".join(f"{k} {v}" for k, v in VERSIONS.items())

### Cell 3 — Nơi lưu kết quả
Colab tắt phiên bất kỳ lúc nào (idle ~90 phút, hết quota ngày) và **xóa toàn bộ đĩa VM**. Vì vậy mọi kết quả
(checkpoint, adapter, log) ghi thẳng lên Google Drive: `OUT = /content/drive/MyDrive/ChatSystem/runs/<RUN_NAME>`.
Kaggle: `/kaggle/working/runs/<RUN_NAME>`; máy cá nhân: `runs/<RUN_NAME>` trong repo (đã gitignore).
Cache model Hugging Face để trên đĩa VM (mặc định) — không cần đưa lên Drive.

`RUN_NAME` trùng notebook 01 để `ADAPTER = $OUT/full` tồn tại.

In [ ]:
# Cell 3 — where run outputs go. Colab: Google Drive (survives a disconnect); Kaggle: /kaggle/working; local: runs/.
RUN_NAME = "xlam2k_qwen05b"   # one folder per experiment; keep the SAME name across notebooks 01/02/03

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/ChatSystem/runs/{RUN_NAME}"
elif IN_KAGGLE:
    OUT = f"/kaggle/working/runs/{RUN_NAME}"
else:
    OUT = str(REPO / "runs" / RUN_NAME)   # runs/ is gitignored
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT   # `!python … $OUT/…` lines and subprocesses see the same path

ADAPTER = f"{OUT}/full"
os.environ["ADAPTER"] = ADAPTER
EVAL = "data/public/xlam_2k.eval.json"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SERVED_NAME = "ft"
PORT = 8000
VLLM_LOG = f"{OUT}/vllm_server.log"
_has_adapter = os.path.exists(f"{ADAPTER}/adapter_config.json")
print("ADAPTER =", ADAPTER, "(found)" if _has_adapter else "(KHÔNG thấy adapter_config.json — kiểm tra RUN_NAME / Drive)")
print("OUT =", OUT)

## Bước 1 — Cài vLLM (vài phút; có thể phải restart session)
Không pin phiên bản. Sau khi cell chạy xong, nếu Colab hiện banner *"You must restart the runtime"* → restart, rồi chạy
lại **Cell 1 → Cell 3** (không chạy lại cell này).

In [ ]:
if IN_COLAB or IN_KAGGLE:
    !pip install -q vllm
else:
    print("Local run: install vllm yourself (needs a CUDA GPU).")
import importlib.metadata as _md
for _p in ("vllm", "torch", "transformers"):
    try:
        print(f"{_p:<13} {_md.version(_p)}")
    except _md.PackageNotFoundError:
        print(f"{_p:<13} not installed")

## Bước 2 — Khởi động vLLM server (tiến trình nền, log ghi vào file)
Đúng các cờ dưới đây (trùng cách phục vụ trên máy GPU của hạ tầng tham chiếu qua `training/serve_vllm.sh`, chỉ khác
`--dtype half` vì T4 không có bf16): `--enable-lora --max-lora-rank 16 --lora-modules ft=$ADAPTER --enable-auto-tool-choice
--tool-call-parser hermes`. Server chạy bằng `subprocess.Popen`, stdout/stderr vào `$OUT/vllm_server.log`.
Nạp model + compile mất 1–5 phút trên T4; cell đợi `GET /v1/models` tối đa 10 phút.

In [ ]:
# Start the vLLM OpenAI-compatible server in the background; poll /v1/models for up to 10 minutes.
import json
import subprocess
import sys
import time
import urllib.request

VLLM_CMD = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL,
    "--dtype", "half",
    "--max-model-len", "4096",
    "--enable-lora", "--max-lora-rank", "16",
    "--lora-modules", f"{SERVED_NAME}={ADAPTER}",
    "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
    "--port", str(PORT),
]
print("$", " ".join(VLLM_CMD))
_vllm_log_fh = open(VLLM_LOG, "a")
vllm_proc = subprocess.Popen(VLLM_CMD, stdout=_vllm_log_fh, stderr=subprocess.STDOUT)
print(f"vLLM pid={vllm_proc.pid} · log → {VLLM_LOG}")


def wait_for_models(url=f"http://127.0.0.1:{PORT}/v1/models", timeout_s=600, every_s=10):
    """Return the /v1/models payload once the server answers; raise if it dies or 10 min pass."""
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if vllm_proc.poll() is not None:
            tail = subprocess.run(["tail", "-n", "40", VLLM_LOG], capture_output=True, text=True).stdout
            raise RuntimeError(f"vLLM exited early with code {vllm_proc.returncode}\n--- last log lines ---\n{tail}")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                return json.load(r)
        except Exception:
            time.sleep(every_s)
            print(f"  waiting… {int(time.time() - t0)} s", end="\r")
    tail = subprocess.run(["tail", "-n", "40", VLLM_LOG], capture_output=True, text=True).stdout
    raise TimeoutError(f"/v1/models not ready after {timeout_s} s\n--- last log lines ---\n{tail}")


models = wait_for_models()
_ids = [m.get("id") for m in models.get("data", [])]
print("\n/v1/models →", _ids)
if SERVED_NAME not in _ids:
    print(f"WARNING: LoRA '{SERVED_NAME}' không có trong /v1/models — kiểm tra ADAPTER và {VLLM_LOG}")

## Bước 3 — Dự đoán 50 bản ghi qua API và chấm điểm
`predict_toolcall.py --backend openai` gửi `chat.completions` với `tools=` tới `--base-url`; vLLM render prompt bằng
chính chat template của Qwen (giống lúc huấn luyện) và parse `<tool_call>` bằng hermes parser. Server local không
cần khóa, nhưng client OpenAI đòi một chuỗi bất kỳ → đặt biến môi trường `DUMMY=x` và trỏ `--api-key-env DUMMY`.

Lưu ý khi đọc kết quả chấm: `eval_toolcall.py` chấm **toàn bộ 200** bản ghi gold; 150 bản ghi không có dự đoán
(`--limit 50`) bị tính `missing_prediction`, nên `accuracy` trong `vllm_results.md` là trên 200. Cell báo cáo tính lại
tỷ lệ **trên 50 bản ghi đã dự đoán** từ chính `by_reason` của scorer (không chấm lại), và ghi rõ điều đó.

In [ ]:
import os
os.environ["DUMMY"] = "x"   # placeholder key for the local server; predict_toolcall reads it via --api-key-env
!python training/predict_toolcall.py --backend openai --base-url http://127.0.0.1:8000/v1 --model ft --api-key-env DUMMY --eval data/public/xlam_2k.eval.json --limit 50 --out $OUT/vllm_pred.jsonl
!echo "----------------------------------------------------------------"
!python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json --pred $OUT/vllm_pred.jsonl --out $OUT/vllm_results.json --md $OUT/vllm_results.md

## Bước 4 — Độ trễ p50 và dừng server
`latency_ms` do `predict_toolcall.py` đo cho từng request (client-side, gồm cả thời gian sinh token). Với 0.5B trên T4
và một request tại một thời điểm, p50 kỳ vọng ở mức **vài trăm ms**; con số bạn đo ra chỉ có ý nghĩa khi ghi kèm
điều kiện (GPU, `--max-model-len`, số bản ghi, `git HEAD`) và, nếu đưa vào báo cáo có số, một hàng `results/INDEX.md`.
Để so sánh định tính: một mô hình 3B phục vụ từ xa hay GPT qua gateway OpenAI-compatible có độ trễ lớn hơn nhiều bậc — xem
`docs/architecture.md` và `docs/serving.md` (mục đo p50); **không** dán số của hệ thống tham chiếu (POC v1) vào báo cáo của bạn.

In [ ]:
# p50 / p95 latency from the predictions file (skip the header line), then stop the server.
import json
import statistics
import subprocess

P50_MS = P95_MS = N_PRED = None
try:
    with open(f"{OUT}/vllm_pred.jsonl") as f:
        rows = [json.loads(ln) for ln in f if ln.strip()]
    lat = sorted(r["latency_ms"] for r in rows if not r.get("header") and r.get("latency_ms") is not None)
    if lat:
        N_PRED = len(lat)
        P50_MS = statistics.median(lat)
        P95_MS = lat[min(len(lat) - 1, int(round(0.95 * (len(lat) - 1))))]
        print(f"n={N_PRED} · p50={P50_MS:.0f} ms · p95={P95_MS:.0f} ms · min={lat[0]:.0f} · max={lat[-1]:.0f}")
    else:
        print("no latency_ms values found")
except FileNotFoundError:
    print(f"{OUT}/vllm_pred.jsonl chưa có — chạy Bước 3 trước.")

try:
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        vllm_proc.kill()
    _vllm_log_fh.close()
    print("vLLM server stopped, exit code", vllm_proc.returncode)
except NameError:
    print("vLLM server was not started in this session.")

## Dự phòng — nếu vLLM không chạy được trên T4
Không cần cố quá một lần restart. **Bằng chứng độ chính xác từ notebook 02 là đủ** cho F Tuần 2; việc phục vụ adapter
bằng vLLM chạy trên máy GPU của hạ tầng tham chiếu theo `docs/serving.md` (và task *F Việc 5* sau này chỉ yêu cầu gói adapter đúng định
dạng PEFT). Ghi vào báo cáo: phiên bản `vllm`/`torch` đã thử, 10 dòng cuối `vllm_server.log`, và dừng ở đó.

## Tạo báo cáo

In [ ]:
# "## Tạo báo cáo" — prints ONE Markdown block to paste as the ClickUp task comment.
# Every number is read from files the scripts wrote; anything missing prints as n/a (never typed by hand).
import datetime
import json
import re
import subprocess


def _read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _grep(path, pattern, max_lines=3):
    """Last `max_lines` lines of `path` matching `pattern` (case-insensitive); [] when the file is missing."""
    try:
        with open(path, errors="replace") as f:
            hits = [ln.rstrip() for ln in f if re.search(pattern, ln, re.I)]
        return hits[-max_lines:]
    except FileNotFoundError:
        return []


def _ls(path):
    r = subprocess.run(["ls", "-la", path], capture_output=True, text=True)
    return r.stdout.strip() or f"(missing: {path})"


def _log_history(path):
    """log_history.json is trainer.state.log_history: a list of dicts (accept {"log_history": [...]} too)."""
    hist = _read_json(path)
    if isinstance(hist, dict):
        hist = hist.get("log_history", [])
    return hist or []


ENV_LINE = "Colab" if IN_COLAB else ("Kaggle" if globals().get("IN_KAGGLE") else "local")
TODAY = datetime.date.today().isoformat()

TASK = "Bước nâng cao (không bắt buộc) — Phục vụ adapter bằng vLLM trên Colab T4"
N_PRED, P50_MS, P95_MS = (globals().get(k) for k in ("N_PRED", "P50_MS", "P95_MS"))  # set in Step 4
res = _read_json(f"{OUT}/vllm_results.json") or {}
by_reason = res.get("by_reason") or {}
n_gold = res.get("n") or len(res.get("results", []))
n_missing = by_reason.get("missing_prediction", 0)
n_scored = n_gold - n_missing                  # records that actually had a prediction (--limit 50)
passed = res.get("passed") if res.get("passed") is not None else sum(1 for r in res.get("results", []) if r.get("pass"))
acc_line = (f"{100.0 * passed / n_scored:.1f}% ({passed}/{n_scored} bản ghi đã dự đoán; scorer báo {res.get('accuracy')}% trên {n_gold} vì {n_missing} bản ghi không được dự đoán)"
            if n_scored else "n/a")
log_tail = subprocess.run(["tail", "-n", "5", VLLM_LOG], capture_output=True, text=True).stdout.strip() or "n/a"
try:
    import importlib.metadata as _md
    vllm_ver = _md.version("vllm")
except Exception:
    vllm_ver = "not installed"
_f = lambda v: f"{v:.0f} ms" if isinstance(v, (int, float)) else "n/a"
reasons = ", ".join(f"{k}={v}" for k, v in by_reason.items() if k not in ("ok", "missing_prediction")) or "—"

lines = [
    f"## Báo cáo {TASK} — {TODAY} — {AUTHOR}",
    f"- Notebook: `notebooks/finetune/03_serve_vllm_colab.ipynb` @ `{GIT_SHA}` · môi trường: {ENV_LINE} · RUN_NAME `{RUN_NAME}` · adapter `{ADAPTER}` served as `{SERVED_NAME}`",
    f"- GPU: {GPU_LINE}",
    f"- Phiên bản: vllm {vllm_ver} · {VERSIONS_LINE}",
    f"- Accuracy qua vLLM API (strict): {acc_line} · fail reasons: {reasons}",
    f"- Latency (client-side, từ vllm_pred.jsonl): n = {N_PRED or 'n/a'} · p50 = {_f(P50_MS)} · p95 = {_f(P95_MS)} · (scorer: latency_p50_ms = {res.get('latency_p50_ms', 'n/a')})",
    f"- Lệnh server: `python -m vllm.entrypoints.openai.api_server --model {BASE_MODEL} --dtype half --max-model-len 4096 --enable-lora --max-lora-rank 16 --lora-modules {SERVED_NAME}={ADAPTER} --enable-auto-tool-choice --tool-call-parser hermes --port {PORT}`",
    "- Lệnh dự đoán / chấm:",
    "```",
    f"python training/predict_toolcall.py --backend openai --base-url http://127.0.0.1:{PORT}/v1 --model {SERVED_NAME} --api-key-env DUMMY --eval {EVAL} --limit 50 --out {OUT}/vllm_pred.jsonl",
    f"python training/eval_toolcall.py --gold {EVAL} --pred {OUT}/vllm_pred.jsonl --out {OUT}/vllm_results.json --md {OUT}/vllm_results.md",
    "```",
    "- 5 dòng cuối `vllm_server.log`:",
    "```", log_tail, "```",
]
print("\n".join(lines))